In [16]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [14]:
!pip install -q langchain-openai langchain-core langchain-community duckduckgo-search requests

In [17]:
pip install -U ddgs

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests

In [18]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('top news in india today')
results

"India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood. Hindustan Times: Stay updated with Hindustan Times for the top India news, world events, and breaking stories. Get real-time updates on politics, wars, live cricket scores, entertainment news, and ... Times of India brings the Latest News & Breaking News Headlines from India & around the World. Read Latest News Today on Sports, Business, Health & Fitness, Bollywood & Entertainment, Blogs ... Stay updated with the latest breaking news, live updates, photos, videos and top trending stories from India and around the world across politics, economy, sports and more on The Economic Times. Read Latest India News Updates from India : Today's Top India News Headlines, Live Breaking News from India, Top news in India, News on Indian.India News , Latest News in India , India News Today , Top News in India

In [19]:
llm = ChatOpenAI()

In [20]:
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ESIP2QuX04GiTTIkm06IfA78IB2jf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0dce8-acf0-7791-9a62-309c4e43f1ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
!pip install -q langchain-classic

In [22]:
from langchain.agents import create_agent
from langchain_classic import hub

In [23]:
#pull the ReAct prompt from langchain hub
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react", #LangSmith Hub se hwchase17 ka react prompt mujhe do.
    dangerously_pull_public_prompt=True
)

prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [31]:
@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city
    """
    url = f'https://api.weatherstack.com/current?access_key=61b5625614eb120f91840beb55f1d03c&query={city}'

    response = requests.get(url)

    return response.json()



In [24]:
# imports

from langchain_classic.agents import create_react_agent, AgentExecutor
from langsmith import Client


In [32]:
#create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm = llm,
    prompt = prompt,
    tools=[search_tool, get_weather_data]
)

In [33]:
#wrap it with agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools = [search_tool, get_weather_data],
    verbose = True #Show me the agent's intermediate steps while it is working.
)

In [34]:
#invoke
response = agent_executor.invoke({"input":"find capital of rajasthan and find its current weather condition"})
print(response)



> Entering new AgentExecutor chain...
We need to first find the capital of Rajasthan and then get the current weather data for that city.
Action: duckduckgo_search
Action Input: "capital of Rajasthan"Jodhpur (Hindi pronunciation: [ˈd͡ʒoːd̪ʱ.pʊr] ⓘ) is the second-largest city of the north-western Indian state of Rajasthan, after its capital Jaipur. As of 2025, the city has a population of 1.6 million. [11] It serves as the administrative headquarters of the Jodhpur district and Jodhpur division. It is the historic capital of the Kingdom of Marwar, founded in 1459 by Rao Jodha, a ... Kota (/ ˈkoʊtə / ⓘ), previously known as Kotah, is the third-largest city of the western Indian state of Rajasthan. [8] It is located about 230 km (143 mi) south of the state capital, Jaipur, on the banks of the Chambal River. As of 2024, with a population of over 1.5 million, it is the third most populous city in Rajasthan, after Jaipur and Jodhpur. [9] It is also India's first and world ... Rajasthan, st

In [35]:
response['output']

'The capital of Rajasthan is Jaipur and the current weather in Jaipur is 33°C with partly cloudy conditions.'